# 📊 Data Analysis Notebook

---

## What is this notebook about?

**In simple terms:** Now that we have clean data, we're going to explore it! We'll look for patterns, create charts, and discover interesting facts about food prices around the world.

This is like being a detective - we're looking for clues in the data that tell us a story about what's been happening with food prices.

---

## Questions We'll Answer:

1. 📈 **How have food prices changed over time?**
2. 🌍 **Which countries have the highest/lowest inflation?**
3. 📅 **Are there seasonal patterns in food prices?**
4. 🔗 **What factors are related to each other?**

---
## Step 1: Load Our Tools

**What's happening?** We're loading Python libraries that help us analyse data and create beautiful charts.

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import os

pd.set_option('display.max_columns', None)
plt.style.use('seaborn-v0_8-whitegrid')

import warnings
warnings.filterwarnings('ignore')

print("✅ All analysis tools loaded successfully!")

---
## Step 2: Load Our Cleaned Data

In [ ]:
df = pd.read_csv('../data/cleaned/food_prices_cleaned.csv', parse_dates=['date'])

print("✅ Data Loaded Successfully!")
print("="*50)
print(f"📊 Dataset size: {df.shape[0]:,} rows, {df.shape[1]} columns")
print(f"📅 Time period: {df['date'].min().strftime('%B %Y')} to {df['date'].max().strftime('%B %Y')}")
print(f"🌍 Countries: {df['country'].nunique()}")

In [ ]:
print("👀 First few rows:")
df.head()

---
## Step 3: Basic Statistics

**What's happening?** We're calculating summary statistics to get an overall picture.

| Statistic | Meaning |
|-----------|--------|
| **mean** | The average |
| **std** | How spread out values are |
| **min/max** | Smallest/largest values |
| **50%** | The middle value (median) |

In [ ]:
print("📊 SUMMARY STATISTICS")
print("="*60)

numerical_cols = ['open', 'high', 'low', 'close', 'inflation', 'price_range', 'price_change_pct']
df[numerical_cols].describe().round(3)

In [ ]:
print("📊 KEY INSIGHTS")
print("="*60)
print(f"\n💰 Average price index: {df['close'].mean():.2f}")
print(f"📈 Average inflation: {df['inflation'].mean():.2f}%")
print(f"📊 Highest inflation recorded: {df['inflation'].max():.2f}%")
print(f"📉 Lowest inflation recorded: {df['inflation'].min():.2f}%")

---
## Step 4: Top and Bottom Countries

In [ ]:
print("🔥 TOP 5 COUNTRIES - HIGHEST AVERAGE INFLATION")
print("="*50)
top5 = df.groupby('country')['inflation'].mean().sort_values(ascending=False).head(5)
for rank, (country, val) in enumerate(top5.items(), 1):
    print(f"   {rank}. {country}: {val:.2f}%")

print("\n🌟 TOP 5 COUNTRIES - LOWEST AVERAGE INFLATION")
print("="*50)
bottom5 = df.groupby('country')['inflation'].mean().sort_values().head(5)
for rank, (country, val) in enumerate(bottom5.items(), 1):
    print(f"   {rank}. {country}: {val:.2f}%")

---
## Step 5: Visualisations

In [ ]:
os.makedirs('../outputs/figures', exist_ok=True)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Distribution of prices
axes[0, 0].hist(df['close'], bins=50, edgecolor='black', alpha=0.7, color='steelblue')
axes[0, 0].set_title('Distribution of Food Prices', fontweight='bold')
axes[0, 0].set_xlabel('Price Index')
axes[0, 0].axvline(df['close'].mean(), color='red', linestyle='--', label=f'Mean: {df["close"].mean():.2f}')
axes[0, 0].legend()

# Distribution of inflation
axes[0, 1].hist(df['inflation'].dropna(), bins=50, edgecolor='black', alpha=0.7, color='coral')
axes[0, 1].set_title('Distribution of Inflation', fontweight='bold')
axes[0, 1].set_xlabel('Inflation Rate (%)')
axes[0, 1].axvline(0, color='black', linestyle='-', label='Zero')
axes[0, 1].legend()

# Volatility distribution
axes[1, 0].hist(df['price_range'], bins=50, edgecolor='black', alpha=0.7, color='mediumpurple')
axes[1, 0].set_title('Distribution of Price Volatility', fontweight='bold')
axes[1, 0].set_xlabel('Price Range')

# Inflation by year
sns.boxplot(data=df[df['inflation'].notna()], x='year', y='inflation', ax=axes[1, 1], palette='viridis')
axes[1, 1].set_title('Inflation by Year', fontweight='bold')
axes[1, 1].tick_params(axis='x', rotation=45)
axes[1, 1].axhline(y=0, color='red', linestyle='--')

plt.tight_layout()
plt.savefig('../outputs/figures/distribution_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

print("💾 Saved: outputs/figures/distribution_analysis.png")

---
## Step 6: Time Series - Trends Over Time

In [ ]:
monthly_avg = df.groupby('date').agg({'close': 'mean', 'inflation': 'mean'}).reset_index()

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

axes[0].plot(monthly_avg['date'], monthly_avg['close'], color='steelblue', linewidth=1.5)
axes[0].fill_between(monthly_avg['date'], monthly_avg['close'], alpha=0.3)
axes[0].set_title('📈 Global Average Food Price Over Time', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Average Price Index')
axes[0].grid(True, alpha=0.3)

axes[1].plot(monthly_avg['date'], monthly_avg['inflation'], color='coral', linewidth=1.5)
axes[1].axhline(y=0, color='black', linestyle='--')
axes[1].fill_between(monthly_avg['date'], monthly_avg['inflation'], alpha=0.3, color='coral')
axes[1].set_title('📊 Global Average Inflation Over Time', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Inflation (%)')
axes[1].set_xlabel('Date')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../outputs/figures/time_series_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

print("💾 Saved: outputs/figures/time_series_analysis.png")

---
## Step 7: Country Comparison

In [ ]:
country_inflation = df.groupby('country')['inflation'].mean().sort_values(ascending=True).dropna()

fig, ax = plt.subplots(figsize=(12, 10))
colors = ['coral' if x > 0 else 'steelblue' for x in country_inflation]
country_inflation.plot(kind='barh', ax=ax, color=colors, edgecolor='black')
ax.axvline(x=0, color='black', linewidth=1)
ax.set_title('🌍 Average Inflation by Country', fontsize=14, fontweight='bold')
ax.set_xlabel('Average Inflation (%)')
ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig('../outputs/figures/country_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

print("💾 Saved: outputs/figures/country_comparison.png")

---
## Step 8: Correlation Analysis

**What is correlation?**
- **+1** = They move together
- **-1** = They move opposite
- **0** = No relationship

In [ ]:
correlation_cols = ['open', 'high', 'low', 'close', 'inflation', 'price_range']
correlation_matrix = df[correlation_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(correlation_matrix, dtype=bool))
sns.heatmap(correlation_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, square=True, ax=ax, vmin=-1, vmax=1)
ax.set_title('🔗 Correlation Heatmap', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('../outputs/figures/correlation_matrix.png', dpi=300, bbox_inches='tight')
plt.show()

print("💾 Saved: outputs/figures/correlation_matrix.png")

---
## Step 9: Seasonal Patterns

In [ ]:
month_names = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
monthly_avg_season = df.groupby('month')['inflation'].mean()

fig, ax = plt.subplots(figsize=(10, 5))
colors = ['coral' if x > monthly_avg_season.mean() else 'steelblue' for x in monthly_avg_season]
monthly_avg_season.plot(kind='bar', ax=ax, color=colors, edgecolor='black')
ax.set_xticklabels(month_names, rotation=45)
ax.set_title('📅 Average Inflation by Month', fontweight='bold')
ax.set_xlabel('Month')
ax.set_ylabel('Average Inflation (%)')
ax.axhline(y=monthly_avg_season.mean(), color='black', linestyle='--', label='Average')
ax.legend()

plt.tight_layout()
plt.savefig('../outputs/figures/seasonal_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

print("💾 Saved: outputs/figures/seasonal_analysis.png")

---
## ✅ Summary

### Key Findings:

1. **Food prices have increased** over the study period
2. **Inflation varies significantly** between countries
3. **Price volatility** appears related to inflation
4. **Some seasonal patterns** may exist

---

**Next step:** Open `Hypothesis_Testing.ipynb` to statistically test these patterns!

# 📊 Data Analysis Notebook

---

## What is this notebook about?

**In simple terms:** Now that we have clean data, we're going to explore it! We'll look for patterns, create charts, and discover interesting facts about food prices around the world.

This is like being a detective - we're looking for clues in the data that tell us a story about what's been happening with food prices.

---

## Questions We'll Answer:

1. 📈 **How have food prices changed over time?**
2. 🌍 **Which countries have the highest/lowest inflation?**
3. 📅 **Are there seasonal patterns in food prices?**
4. 🔗 **What factors are related to each other?**

---
## Step 1: Load Our Tools

**What's happening?** We're loading Python libraries that help us analyse data and create beautiful charts.

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import os

pd.set_option('display.max_columns', None)
plt.style.use('seaborn-v0_8-whitegrid')

import warnings
warnings.filterwarnings('ignore')

print("✅ All analysis tools loaded successfully!")

---
## Step 2: Load Our Cleaned Data

In [ ]:
df = pd.read_csv('../data/cleaned/food_prices_cleaned.csv', parse_dates=['date'])

print("✅ Data Loaded Successfully!")
print("="*50)
print(f"📊 Dataset size: {df.shape[0]:,} rows, {df.shape[1]} columns")
print(f"📅 Time period: {df['date'].min().strftime('%B %Y')} to {df['date'].max().strftime('%B %Y')}")
print(f"🌍 Countries: {df['country'].nunique()}")

In [ ]:
print("👀 First few rows:")
df.head()

---
## Step 3: Basic Statistics

**What's happening?** We're calculating summary statistics to get an overall picture.

| Statistic | Meaning |
|-----------|--------|
| **mean** | The average |
| **std** | How spread out values are |
| **min/max** | Smallest/largest values |
| **50%** | The middle value (median) |

In [ ]:
print("📊 SUMMARY STATISTICS")
print("="*60)

numerical_cols = ['open', 'high', 'low', 'close', 'inflation', 'price_range', 'price_change_pct']
df[numerical_cols].describe().round(3)

In [ ]:
print("📊 KEY INSIGHTS")
print("="*60)
print(f"\n💰 Average price index: {df['close'].mean():.2f}")
print(f"📈 Average inflation: {df['inflation'].mean():.2f}%")
print(f"📊 Highest inflation recorded: {df['inflation'].max():.2f}%")
print(f"📉 Lowest inflation recorded: {df['inflation'].min():.2f}%")

---
## Step 4: Top and Bottom Countries

In [ ]:
print("🔥 TOP 5 COUNTRIES - HIGHEST AVERAGE INFLATION")
print("="*50)
top5 = df.groupby('country')['inflation'].mean().sort_values(ascending=False).head(5)
for rank, (country, val) in enumerate(top5.items(), 1):
    print(f"   {rank}. {country}: {val:.2f}%")

print("\n🌟 TOP 5 COUNTRIES - LOWEST AVERAGE INFLATION")
print("="*50)
bottom5 = df.groupby('country')['inflation'].mean().sort_values().head(5)
for rank, (country, val) in enumerate(bottom5.items(), 1):
    print(f"   {rank}. {country}: {val:.2f}%")

---
## Step 5: Visualisations

In [ ]:
os.makedirs('../outputs/figures', exist_ok=True)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Distribution of prices
axes[0, 0].hist(df['close'], bins=50, edgecolor='black', alpha=0.7, color='steelblue')
axes[0, 0].set_title('Distribution of Food Prices', fontweight='bold')
axes[0, 0].set_xlabel('Price Index')
axes[0, 0].axvline(df['close'].mean(), color='red', linestyle='--', label=f'Mean: {df["close"].mean():.2f}')
axes[0, 0].legend()

# Distribution of inflation
axes[0, 1].hist(df['inflation'].dropna(), bins=50, edgecolor='black', alpha=0.7, color='coral')
axes[0, 1].set_title('Distribution of Inflation', fontweight='bold')
axes[0, 1].set_xlabel('Inflation Rate (%)')
axes[0, 1].axvline(0, color='black', linestyle='-', label='Zero')
axes[0, 1].legend()

# Volatility distribution
axes[1, 0].hist(df['price_range'], bins=50, edgecolor='black', alpha=0.7, color='mediumpurple')
axes[1, 0].set_title('Distribution of Price Volatility', fontweight='bold')
axes[1, 0].set_xlabel('Price Range')

# Inflation by year
sns.boxplot(data=df[df['inflation'].notna()], x='year', y='inflation', ax=axes[1, 1], palette='viridis')
axes[1, 1].set_title('Inflation by Year', fontweight='bold')
axes[1, 1].tick_params(axis='x', rotation=45)
axes[1, 1].axhline(y=0, color='red', linestyle='--')

plt.tight_layout()
plt.savefig('../outputs/figures/distribution_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

print("💾 Saved: outputs/figures/distribution_analysis.png")

---
## Step 6: Time Series - Trends Over Time

In [ ]:
monthly_avg = df.groupby('date').agg({'close': 'mean', 'inflation': 'mean'}).reset_index()

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

axes[0].plot(monthly_avg['date'], monthly_avg['close'], color='steelblue', linewidth=1.5)
axes[0].fill_between(monthly_avg['date'], monthly_avg['close'], alpha=0.3)
axes[0].set_title('📈 Global Average Food Price Over Time', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Average Price Index')
axes[0].grid(True, alpha=0.3)

axes[1].plot(monthly_avg['date'], monthly_avg['inflation'], color='coral', linewidth=1.5)
axes[1].axhline(y=0, color='black', linestyle='--')
axes[1].fill_between(monthly_avg['date'], monthly_avg['inflation'], alpha=0.3, color='coral')
axes[1].set_title('📊 Global Average Inflation Over Time', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Inflation (%)')
axes[1].set_xlabel('Date')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../outputs/figures/time_series_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

print("💾 Saved: outputs/figures/time_series_analysis.png")

---
## Step 7: Country Comparison

In [ ]:
country_inflation = df.groupby('country')['inflation'].mean().sort_values(ascending=True).dropna()

fig, ax = plt.subplots(figsize=(12, 10))
colors = ['coral' if x > 0 else 'steelblue' for x in country_inflation]
country_inflation.plot(kind='barh', ax=ax, color=colors, edgecolor='black')
ax.axvline(x=0, color='black', linewidth=1)
ax.set_title('🌍 Average Inflation by Country', fontsize=14, fontweight='bold')
ax.set_xlabel('Average Inflation (%)')
ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig('../outputs/figures/country_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

print("💾 Saved: outputs/figures/country_comparison.png")

---
## Step 8: Correlation Analysis

**What is correlation?**
- **+1** = They move together
- **-1** = They move opposite
- **0** = No relationship

In [ ]:
correlation_cols = ['open', 'high', 'low', 'close', 'inflation', 'price_range']
correlation_matrix = df[correlation_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(correlation_matrix, dtype=bool))
sns.heatmap(correlation_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, square=True, ax=ax, vmin=-1, vmax=1)
ax.set_title('🔗 Correlation Heatmap', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('../outputs/figures/correlation_matrix.png', dpi=300, bbox_inches='tight')
plt.show()

print("💾 Saved: outputs/figures/correlation_matrix.png")

---
## Step 9: Seasonal Patterns

In [ ]:
month_names = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
monthly_avg_season = df.groupby('month')['inflation'].mean()

fig, ax = plt.subplots(figsize=(10, 5))
colors = ['coral' if x > monthly_avg_season.mean() else 'steelblue' for x in monthly_avg_season]
monthly_avg_season.plot(kind='bar', ax=ax, color=colors, edgecolor='black')
ax.set_xticklabels(month_names, rotation=45)
ax.set_title('📅 Average Inflation by Month', fontweight='bold')
ax.set_xlabel('Month')
ax.set_ylabel('Average Inflation (%)')
ax.axhline(y=monthly_avg_season.mean(), color='black', linestyle='--', label='Average')
ax.legend()

plt.tight_layout()
plt.savefig('../outputs/figures/seasonal_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

print("💾 Saved: outputs/figures/seasonal_analysis.png")

---
## ✅ Summary

### Key Findings:

1. **Food prices have increased** over the study period
2. **Inflation varies significantly** between countries
3. **Price volatility** appears related to inflation
4. **Some seasonal patterns** may exist

---

**Next step:** Open `Hypothesis_Testing.ipynb` to statistically test these patterns!

# 📊 Data Analysis Notebook

---

## What is this notebook about?

**In simple terms:** Now that we have clean data, we're going to explore it! We'll look for patterns, create charts, and discover interesting facts about food prices around the world.

This is like being a detective - we're looking for clues in the data that tell us a story about what's been happening with food prices.

---

## Questions We'll Answer:

1. 📈 **How have food prices changed over time?**
2. 🌍 **Which countries have the highest/lowest inflation?**
3. 📅 **Are there seasonal patterns in food prices?**
4. 🔗 **What factors are related to each other?**

---

## What You'll Learn:

- How to calculate statistics (averages, ranges, etc.)
- How to create visualisations (charts and graphs)
- How to interpret data patterns
- How to draw meaningful conclusions from data

---
## Step 1: Load Our Tools

**What's happening?** We're loading Python libraries that help us analyse data and create beautiful charts.

Think of these as specialised equipment:
- **pandas** - For working with data tables
- **numpy** - For mathematical calculations  
- **matplotlib & seaborn** - For creating static charts
- **plotly** - For creating interactive charts you can click and explore
- **scipy** - For statistical calculations

In [ ]:
# Core data libraries
import pandas as pd
import numpy as np

# Statistical library
from scipy import stats

# Visualisation libraries
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Display settings - make our output look nice
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
plt.style.use('seaborn-v0_8-whitegrid')

# Hide warning messages to keep output clean
import warnings
warnings.filterwarnings('ignore')

import os

print("✅ All analysis tools loaded successfully!")

---
## Step 2: Load Our Cleaned Data

**What's happening?** We're loading the clean data that we prepared in the Data Cleaning notebook.

In [ ]:
# Load the cleaned dataset
df = pd.read_csv('../data/cleaned/food_prices_cleaned.csv', parse_dates=['date'])

print("✅ Data Loaded Successfully!")
print("="*50)
print(f"📊 Dataset size: {df.shape[0]:,} rows, {df.shape[1]} columns")
print(f"📅 Time period: {df['date'].min().strftime('%B %Y')} to {df['date'].max().strftime('%B %Y')}")
print(f"🌍 Countries: {df['country'].nunique()}")

In [ ]:
# Quick preview of the data
print("👀 First few rows of our data:")
df.head()

---
## Step 3: Basic Statistics

**What's happening?** We're calculating summary statistics to get an overall picture of our data.

### What do these statistics mean?

| Statistic | Plain English Meaning |
|-----------|----------------------|
| **count** | How many values we have |
| **mean** | The average (add all values and divide by count) |
| **std** | Standard deviation - how spread out the values are |
| **min** | The smallest value |
| **25%** | 25% of values are below this (Q1) |
| **50%** | The middle value (median) - half are above, half below |
| **75%** | 75% of values are below this (Q3) |
| **max** | The largest value |

In [ ]:
# Get statistics for our main numeric columns
print("📊 SUMMARY STATISTICS")
print("="*60)
print("\nThese numbers give us an overview of our data:")
print()

numerical_cols = ['open', 'high', 'low', 'close', 'inflation', 'price_range', 'price_change', 'price_change_pct']
df[numerical_cols].describe().round(3)

### 💡 Key Takeaways from Statistics:

Let's interpret what these numbers tell us:

In [ ]:
print("📊 WHAT THE NUMBERS TELL US")
print("="*60)

print("\n💰 PRICE INDEX (close column):")
print(f"   • Average price index: {df['close'].mean():.2f}")
print(f"   • Prices ranged from {df['close'].min():.2f} to {df['close'].max():.2f}")
print(f"   • This is a {((df['close'].max() / df['close'].min()) - 1) * 100:.0f}% difference between lowest and highest!")

print("\n📈 INFLATION:")
avg_inflation = df['inflation'].mean()
print(f"   • Average inflation: {avg_inflation:.2f}%")
print(f"   • Lowest inflation: {df['inflation'].min():.2f}% (prices dropped!)")
print(f"   • Highest inflation: {df['inflation'].max():.2f}% (prices surged!)")

print("\n📊 PRICE VOLATILITY (how much prices jump around):")
print(f"   • Average monthly price swing: {df['price_range'].mean():.4f}")
print(f"   • Biggest swing in one month: {df['price_range'].max():.4f}")

---
## Step 4: Statistics by Country

**What's happening?** Let's see how different countries compare. Which countries have the highest prices? The most inflation?

In [ ]:
# Calculate average statistics for each country
country_stats = df.groupby('country').agg({
    'close': ['mean', 'std', 'min', 'max'],
    'inflation': ['mean', 'std', 'min', 'max'],
    'price_range': ['mean', 'max'],
    'date': 'count'
}).round(3)

# Make column names easier to read
country_stats.columns = ['_'.join(col).strip() for col in country_stats.columns.values]
country_stats = country_stats.rename(columns={'date_count': 'total_records'})

print("🌍 Statistics by Country:")
print("(Sorted by average inflation, highest first)")
print()
country_stats.sort_values('inflation_mean', ascending=False)

### 🏆 Top 5 Countries with Highest Average Inflation

In [ ]:
# Get top 5 countries by inflation
top_inflation = df.groupby('country')['inflation'].mean().sort_values(ascending=False).head(5)

print("🔥 TOP 5 COUNTRIES WITH HIGHEST AVERAGE INFLATION")
print("="*50)
print("\nThese countries experienced the biggest price increases:")
print()
for rank, (country, inflation) in enumerate(top_inflation.items(), 1):
    print(f"   {rank}. {country}: {inflation:.2f}% average inflation")

### 🥇 Top 5 Countries with Lowest Average Inflation

In [ ]:
# Get bottom 5 countries by inflation
bottom_inflation = df.groupby('country')['inflation'].mean().sort_values().head(5)

print("🌟 TOP 5 COUNTRIES WITH LOWEST AVERAGE INFLATION")
print("="*50)
print("\nThese countries had the most stable (or even decreasing) prices:")
print()
for rank, (country, inflation) in enumerate(bottom_inflation.items(), 1):
    print(f"   {rank}. {country}: {inflation:.2f}% average inflation")

---
## Step 5: Visualisations

**What's happening?** Now we'll create charts to make our data easier to understand. A picture is worth a thousand numbers!

### 5.1 Distribution Charts

**What is a distribution?** It shows how values are spread out. Are most values clustered together, or spread out widely?

In [ ]:
# Create output folder for saving our charts
os.makedirs('../outputs/figures', exist_ok=True)

# Create a figure with 4 charts
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Chart 1: Distribution of Close Prices
axes[0, 0].hist(df['close'], bins=50, edgecolor='black', alpha=0.7, color='steelblue')
axes[0, 0].set_title('How are food prices distributed?', fontsize=12, fontweight='bold')
axes[0, 0].set_xlabel('Close Price Index')
axes[0, 0].set_ylabel('Number of Records')
axes[0, 0].axvline(df['close'].mean(), color='red', linestyle='--', linewidth=2, label=f'Average: {df["close"].mean():.2f}')
axes[0, 0].legend()

# Chart 2: Distribution of Inflation
inflation_data = df['inflation'].dropna()
axes[0, 1].hist(inflation_data, bins=50, edgecolor='black', alpha=0.7, color='coral')
axes[0, 1].set_title('How is inflation distributed?', fontsize=12, fontweight='bold')
axes[0, 1].set_xlabel('Inflation Rate (%)')
axes[0, 1].set_ylabel('Number of Records')
axes[0, 1].axvline(0, color='black', linestyle='-', linewidth=1, label='Zero inflation')
axes[0, 1].axvline(inflation_data.mean(), color='red', linestyle='--', linewidth=2, label=f'Average: {inflation_data.mean():.2f}%')
axes[0, 1].legend()

# Chart 3: Price Volatility Distribution
axes[1, 0].hist(df['price_range'], bins=50, edgecolor='black', alpha=0.7, color='mediumpurple')
axes[1, 0].set_title('How volatile are prices?', fontsize=12, fontweight='bold')
axes[1, 0].set_xlabel('Price Range (High - Low)')
axes[1, 0].set_ylabel('Number of Records')

# Chart 4: Inflation by Year (Box Plot)
df_with_inflation = df[df['inflation'].notna()]
year_order = sorted(df_with_inflation['year'].unique())
sns.boxplot(data=df_with_inflation, x='year', y='inflation', ax=axes[1, 1], palette='viridis')
axes[1, 1].set_title('Inflation spread in each year', fontsize=12, fontweight='bold')
axes[1, 1].set_xlabel('Year')
axes[1, 1].set_ylabel('Inflation (%)')
axes[1, 1].tick_params(axis='x', rotation=45)
axes[1, 1].axhline(y=0, color='red', linestyle='--', linewidth=1)

plt.tight_layout()
plt.savefig('../outputs/figures/distribution_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n💾 Chart saved to: outputs/figures/distribution_analysis.png")

### 💡 What do these charts tell us?

1. **Price Distribution (Top Left):** Most food prices cluster around the average, but there are some very high values on the right side.

2. **Inflation Distribution (Top Right):** Most inflation values are clustered around 0-20%, but there are some extreme values on both ends.

3. **Volatility Distribution (Bottom Left):** Most months have low volatility, but occasionally prices swing wildly.

4. **Inflation by Year (Bottom Right):** Each box shows the range of inflation for that year. We can see some years had more extreme inflation than others.

### 5.2 Time Series Charts - How Have Prices Changed Over Time?

**What is a time series?** It's data measured at regular intervals over time - perfect for seeing trends!

In [ ]:
# Calculate global monthly averages
monthly_avg = df.groupby('date').agg({
    'close': 'mean',
    'inflation': 'mean',
    'price_range': 'mean'
}).reset_index()

# Create 3 time series charts
fig, axes = plt.subplots(3, 1, figsize=(14, 12), sharex=True)

# Chart 1: Average Price Over Time
axes[0].plot(monthly_avg['date'], monthly_avg['close'], color='steelblue', linewidth=1.5)
axes[0].fill_between(monthly_avg['date'], monthly_avg['close'], alpha=0.3)
axes[0].set_title('📈 Global Average Food Price Over Time', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Average Price Index')
axes[0].grid(True, alpha=0.3)

# Chart 2: Average Inflation Over Time
axes[1].plot(monthly_avg['date'], monthly_avg['inflation'], color='coral', linewidth=1.5)
axes[1].axhline(y=0, color='black', linestyle='--', linewidth=1)
axes[1].fill_between(monthly_avg['date'], monthly_avg['inflation'], alpha=0.3, color='coral')
axes[1].set_title('📊 Global Average Inflation Rate Over Time', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Inflation Rate (%)')
axes[1].grid(True, alpha=0.3)

# Chart 3: Volatility Over Time
axes[2].plot(monthly_avg['date'], monthly_avg['price_range'], color='mediumpurple', linewidth=1.5)
axes[2].set_title('📉 Price Volatility Over Time', fontsize=14, fontweight='bold')
axes[2].set_xlabel('Date')
axes[2].set_ylabel('Average Price Swing')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../outputs/figures/time_series_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n💾 Chart saved to: outputs/figures/time_series_analysis.png")

### 💡 What do these time series tell us?

1. **Top Chart (Prices):** Food prices have generally increased over time - you can see the upward trend from 2007 to 2023.

2. **Middle Chart (Inflation):** Inflation fluctuates a lot! Notice the big spikes - these are periods of food price crises. When the line is below zero, prices were actually falling compared to the previous year.

3. **Bottom Chart (Volatility):** Price volatility also changes over time. Some periods are calm (low volatility), while others are turbulent (high volatility).

### 5.3 Country Comparison - Which Countries Have the Highest Inflation?

In [ ]:
# Average inflation by country
country_inflation = df.groupby('country')['inflation'].mean().sort_values(ascending=True).dropna()

# Create horizontal bar chart
fig, ax = plt.subplots(figsize=(12, 10))

# Color bars based on positive/negative inflation
colors = ['coral' if x > 0 else 'steelblue' for x in country_inflation]

country_inflation.plot(kind='barh', ax=ax, color=colors, edgecolor='black')
ax.axvline(x=0, color='black', linewidth=1)
ax.set_title('🌍 Average Inflation Rate by Country (2007-2023)', fontsize=14, fontweight='bold')
ax.set_xlabel('Average Inflation Rate (%)')
ax.set_ylabel('Country')
ax.grid(axis='x', alpha=0.3)

# Add a text box explaining the colors
ax.text(0.02, 0.98, 'Orange = Positive inflation (prices rising)\nBlue = Negative inflation (prices falling)', 
        transform=ax.transAxes, fontsize=10, verticalalignment='top',
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.savefig('../outputs/figures/country_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n💾 Chart saved to: outputs/figures/country_comparison.png")

### 5.4 Interactive Chart - Explore Price Trends by Country

**What's special about this chart?** You can interact with it! Click on country names in the legend to show/hide them, hover over lines to see exact values, and zoom in on interesting periods.

In [ ]:
# Create an interactive line chart
fig = px.line(
    df, 
    x='date', 
    y='close', 
    color='country',
    title='🌍 Food Price Index by Country Over Time (Interactive!)',
    labels={'close': 'Price Index', 'date': 'Date', 'country': 'Country'}
)

fig.update_layout(
    height=600,
    legend=dict(orientation='h', yanchor='bottom', y=-0.3, xanchor='center', x=0.5),
    hovermode='x unified'
)

fig.show()

print("\n💡 Tip: Click on country names in the legend to show/hide them!")
print("   You can also zoom in by clicking and dragging on the chart.")

### 5.5 Correlation Analysis - What Factors Are Related?

**What is correlation?** It measures how two things move together:
- **Positive correlation (+1):** When one goes up, the other goes up too
- **Negative correlation (-1):** When one goes up, the other goes down
- **No correlation (0):** They don't affect each other

In [ ]:
# Calculate correlations between our numerical variables
correlation_cols = ['open', 'high', 'low', 'close', 'inflation', 'price_range', 'price_change_pct']
correlation_matrix = df[correlation_cols].corr()

# Create a heatmap
fig, ax = plt.subplots(figsize=(10, 8))

# Only show the lower triangle to avoid repetition
mask = np.triu(np.ones_like(correlation_matrix, dtype=bool))

sns.heatmap(
    correlation_matrix,
    mask=mask,
    annot=True,
    fmt='.2f',
    cmap='RdBu_r',
    center=0,
    square=True,
    linewidths=0.5,
    ax=ax,
    vmin=-1, vmax=1
)

ax.set_title('🔗 How Are Our Variables Related? (Correlation Heatmap)', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('../outputs/figures/correlation_matrix.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n💾 Chart saved to: outputs/figures/correlation_matrix.png")

### 💡 How to Read This Heatmap:

- **Red = Strong positive correlation** (they move together)
- **Blue = Strong negative correlation** (they move opposite)
- **White = No correlation** (they're not related)

**Key Observations:**
- Open, High, Low, and Close prices are very strongly correlated (dark red) - makes sense, they're all price measures!
- Price volatility (price_range) and inflation show some relationship
- The exact correlation numbers are shown in each cell

### 5.6 Seasonal Patterns - Do Prices Change by Month?

**What's happening?** We're checking if food prices tend to be higher or lower in certain months of the year.

In [ ]:
# Calculate average values for each month
monthly_seasonal = df.groupby('month').agg({
    'close': 'mean',
    'inflation': 'mean',
    'price_range': 'mean'
}).round(3)

# Add month names
month_names = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
monthly_seasonal.index = month_names

# Create 3 bar charts
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Chart 1: Average price by month
monthly_seasonal['close'].plot(kind='bar', ax=axes[0], color='steelblue', edgecolor='black')
axes[0].set_title('Average Price by Month', fontweight='bold')
axes[0].set_xlabel('Month')
axes[0].set_ylabel('Average Price Index')
axes[0].tick_params(axis='x', rotation=45)

# Chart 2: Average inflation by month
monthly_seasonal['inflation'].plot(kind='bar', ax=axes[1], color='coral', edgecolor='black')
axes[1].set_title('Average Inflation by Month', fontweight='bold')
axes[1].set_xlabel('Month')
axes[1].set_ylabel('Average Inflation (%)')
axes[1].tick_params(axis='x', rotation=45)

# Chart 3: Average volatility by month
monthly_seasonal['price_range'].plot(kind='bar', ax=axes[2], color='mediumpurple', edgecolor='black')
axes[2].set_title('Average Volatility by Month', fontweight='bold')
axes[2].set_xlabel('Month')
axes[2].set_ylabel('Average Price Swing')
axes[2].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('../outputs/figures/seasonal_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n💾 Chart saved to: outputs/figures/seasonal_analysis.png")
print("\n📊 Monthly Averages:")
print(monthly_seasonal)

---
## Step 6: Key Insights Summary

**What's happening?** Let's pull together all our findings into clear, actionable insights.

In [ ]:
print("="*70)
print("📋 KEY INSIGHTS FROM OUR DATA ANALYSIS")
print("="*70)

# 1. Overall Statistics
print("\n1️⃣ OVERALL PICTURE")
print("-"*50)
print(f"   • We analysed {len(df):,} records across {df['country'].nunique()} countries")
print(f"   • Time period: {(df['date'].max() - df['date'].min()).days // 365}+ years")
print(f"   • Average food price index: {df['close'].mean():.2f}")
print(f"   • Average inflation rate: {df['inflation'].mean():.1f}%")

# 2. Price Trends
print("\n2️⃣ PRICE TRENDS")
print("-"*50)
first_year = df[df['year'] == df['year'].min()]['close'].mean()
last_year = df[df['year'] == df['year'].max()]['close'].mean()
total_change = ((last_year - first_year) / first_year) * 100
print(f"   • Prices have {'INCREASED' if total_change > 0 else 'DECREASED'} by {abs(total_change):.1f}% overall")
print(f"   • Starting average (2007): {first_year:.2f}")
print(f"   • Ending average (2023): {last_year:.2f}")

# 3. Top and Bottom Countries
print("\n3️⃣ COUNTRY HIGHLIGHTS")
print("-"*50)
print("   Highest average inflation:")
top3 = df.groupby('country')['inflation'].mean().sort_values(ascending=False).head(3)
for i, (country, val) in enumerate(top3.items(), 1):
    print(f"      {i}. {country}: {val:.1f}%")

print("\n   Lowest average inflation:")
bottom3 = df.groupby('country')['inflation'].mean().sort_values().head(3)
for i, (country, val) in enumerate(bottom3.items(), 1):
    print(f"      {i}. {country}: {val:.1f}%")

# 4. Correlations
print("\n4️⃣ KEY RELATIONSHIPS")
print("-"*50)
price_inflation_corr = df['close'].corr(df['inflation'])
vol_inflation_corr = df['price_range'].corr(df['inflation'])
print(f"   • Price level & inflation correlation: {price_inflation_corr:.2f}")
print(f"   • Volatility & inflation correlation: {vol_inflation_corr:.2f}")

# 5. Volatility
print("\n5️⃣ PRICE STABILITY")
print("-"*50)
most_volatile = df.groupby('country')['price_range'].mean().sort_values(ascending=False).head(3)
print("   Most volatile markets (biggest price swings):")
for i, (country, vol) in enumerate(most_volatile.items(), 1):
    print(f"      {i}. {country}")

print("\n" + "="*70)

---
## 📝 Summary: What We Learned

### In Plain English:

1. **Food prices have gone up significantly** over the 16-year period we studied.

2. **Some countries suffer more than others** - inflation varies greatly depending on where you are.

3. **Price volatility matters** - when prices jump around a lot, inflation tends to be higher too.

4. **There are global patterns** - many countries experience similar trends, suggesting worldwide factors affect food prices.

---

### Next Steps:

➡️ Open **Hypothesis_Testing.ipynb** to statistically test whether these patterns are real or could be due to chance.

---

### Charts We Created:

| Chart | File Location | What It Shows |
|-------|--------------|---------------|
| Distribution Analysis | `outputs/figures/distribution_analysis.png` | How values are spread out |
| Time Series | `outputs/figures/time_series_analysis.png` | Changes over time |
| Country Comparison | `outputs/figures/country_comparison.png` | Inflation by country |
| Correlation Heatmap | `outputs/figures/correlation_matrix.png` | Relationships between variables |
| Seasonal Analysis | `outputs/figures/seasonal_analysis.png` | Monthly patterns |

---
## 🤖 AI Assistance Note

This notebook was created with assistance from **GitHub Copilot** (an AI coding assistant). The AI helped with:
- Writing code for data analysis and visualisation
- Suggesting appropriate statistical methods
- Creating clear explanations for non-technical readers
- Optimising chart designs for clarity

All code was reviewed and tested by the analyst to ensure accuracy and appropriateness.

# Data Analysis Notebook

## Objective
Perform exploratory data analysis (EDA) on the cleaned food price inflation dataset to identify patterns, trends, and insights.

## Learning Outcomes Addressed
- **LO1**: Apply core principles of statistics and probability
- **LO2**: Demonstrate practical data manipulation skills with Python
- **LO3**: Analyse real-world problems using data analytics
- **LO4**: Utilise Jupyter Notebooks enhanced by AI assistance
- **LO8**: Communicate complex insights effectively

## Dataset
- **Source**: `data/cleaned/food_prices_cleaned.csv`
- **Records**: 4,798 observations
- **Time Period**: January 2007 - October 2023

---
## 1. Import Libraries

In [ ]:
# Core libraries
import pandas as pd
import numpy as np

# Statistical libraries
from scipy import stats

# Visualisation libraries
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
plt.style.use('seaborn-v0_8-whitegrid')

# Suppress warnings
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully!")

---
## 2. Load Cleaned Data

In [ ]:
# Load the cleaned dataset
df = pd.read_csv('../data/cleaned/food_prices_cleaned.csv', parse_dates=['date'])

print(f"Dataset loaded: {df.shape[0]} rows, {df.shape[1]} columns")
print(f"\nDate Range: {df['date'].min()} to {df['date'].max()}")
print(f"Countries: {df['country'].nunique()}")

In [ ]:
# Preview the data
df.head(10)

In [ ]:
# Data types
df.dtypes

---
## 3. Descriptive Statistics

### 3.1 Overall Summary Statistics

In [ ]:
# Statistical summary of numerical columns
numerical_cols = ['open', 'high', 'low', 'close', 'inflation', 'price_range', 'price_change', 'price_change_pct']
df[numerical_cols].describe()

In [ ]:
# Additional statistics: skewness and kurtosis
stats_df = pd.DataFrame({
    'Count': df[numerical_cols].count(),
    'Mean': df[numerical_cols].mean(),
    'Median': df[numerical_cols].median(),
    'Std Dev': df[numerical_cols].std(),
    'Skewness': df[numerical_cols].skew(),
    'Kurtosis': df[numerical_cols].kurtosis(),
    'Min': df[numerical_cols].min(),
    'Max': df[numerical_cols].max()
})

print("Extended Statistical Summary:")
stats_df.round(3)

### 3.2 Statistics by Country

In [ ]:
# Summary statistics by country
country_stats = df.groupby('country').agg({
    'close': ['mean', 'std', 'min', 'max'],
    'inflation': ['mean', 'std', 'min', 'max'],
    'price_range': ['mean', 'max'],
    'date': 'count'
}).round(3)

# Flatten column names
country_stats.columns = ['_'.join(col).strip() for col in country_stats.columns.values]
country_stats = country_stats.rename(columns={'date_count': 'observations'})

print("Statistics by Country:")
country_stats.sort_values('inflation_mean', ascending=False)

### 3.3 Statistics by Year

In [ ]:
# Yearly trends
yearly_stats = df.groupby('year').agg({
    'close': 'mean',
    'inflation': 'mean',
    'price_range': 'mean',
    'country': 'nunique'
}).round(3)

yearly_stats.columns = ['avg_close', 'avg_inflation', 'avg_price_range', 'countries_count']
yearly_stats

---
## 4. Data Visualisations

### 4.1 Distribution Analysis

In [ ]:
# Distribution of key variables
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Close price distribution
axes[0, 0].hist(df['close'], bins=50, edgecolor='black', alpha=0.7, color='steelblue')
axes[0, 0].set_title('Distribution of Close Prices', fontsize=12)
axes[0, 0].set_xlabel('Close Price')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].axvline(df['close'].mean(), color='red', linestyle='--', label=f'Mean: {df["close"].mean():.2f}')
axes[0, 0].axvline(df['close'].median(), color='green', linestyle='--', label=f'Median: {df["close"].median():.2f}')
axes[0, 0].legend()

# Inflation distribution
inflation_data = df['inflation'].dropna()
axes[0, 1].hist(inflation_data, bins=50, edgecolor='black', alpha=0.7, color='coral')
axes[0, 1].set_title('Distribution of Inflation Rates', fontsize=12)
axes[0, 1].set_xlabel('Inflation (%)')
axes[0, 1].set_ylabel('Frequency')
axes[0, 1].axvline(inflation_data.mean(), color='red', linestyle='--', label=f'Mean: {inflation_data.mean():.2f}%')
axes[0, 1].legend()

# Price range distribution
axes[1, 0].hist(df['price_range'], bins=50, edgecolor='black', alpha=0.7, color='mediumpurple')
axes[1, 0].set_title('Distribution of Price Range (Volatility)', fontsize=12)
axes[1, 0].set_xlabel('Price Range (High - Low)')
axes[1, 0].set_ylabel('Frequency')

# Box plot of inflation by year
df_with_inflation = df[df['inflation'].notna()]
sns.boxplot(data=df_with_inflation, x='year', y='inflation', ax=axes[1, 1], palette='viridis')
axes[1, 1].set_title('Inflation Distribution by Year', fontsize=12)
axes[1, 1].set_xlabel('Year')
axes[1, 1].set_ylabel('Inflation (%)')
axes[1, 1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('../outputs/figures/distribution_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

print("Figure saved to: outputs/figures/distribution_analysis.png")

### 4.2 Time Series Analysis

In [ ]:
# Global average price over time
monthly_avg = df.groupby('date').agg({
    'close': 'mean',
    'inflation': 'mean',
    'price_range': 'mean'
}).reset_index()

fig, axes = plt.subplots(3, 1, figsize=(14, 12), sharex=True)

# Average close price
axes[0].plot(monthly_avg['date'], monthly_avg['close'], color='steelblue', linewidth=1.5)
axes[0].fill_between(monthly_avg['date'], monthly_avg['close'], alpha=0.3)
axes[0].set_title('Global Average Food Price Index Over Time', fontsize=14)
axes[0].set_ylabel('Average Close Price')
axes[0].grid(True, alpha=0.3)

# Average inflation
axes[1].plot(monthly_avg['date'], monthly_avg['inflation'], color='coral', linewidth=1.5)
axes[1].axhline(y=0, color='black', linestyle='--', linewidth=0.5)
axes[1].fill_between(monthly_avg['date'], monthly_avg['inflation'], alpha=0.3, color='coral')
axes[1].set_title('Global Average Inflation Rate Over Time', fontsize=14)
axes[1].set_ylabel('Average Inflation (%)')
axes[1].grid(True, alpha=0.3)

# Price volatility (range)
axes[2].plot(monthly_avg['date'], monthly_avg['price_range'], color='mediumpurple', linewidth=1.5)
axes[2].set_title('Global Average Price Volatility Over Time', fontsize=14)
axes[2].set_xlabel('Date')
axes[2].set_ylabel('Average Price Range')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../outputs/figures/time_series_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

print("Figure saved to: outputs/figures/time_series_analysis.png")

### 4.3 Country Comparison

In [ ]:
# Average inflation by country
country_inflation = df.groupby('country')['inflation'].mean().sort_values(ascending=True).dropna()

fig, ax = plt.subplots(figsize=(12, 10))
colors = ['coral' if x > 0 else 'steelblue' for x in country_inflation]
country_inflation.plot(kind='barh', ax=ax, color=colors, edgecolor='black')
ax.axvline(x=0, color='black', linewidth=0.5)
ax.set_title('Average Inflation Rate by Country (2007-2023)', fontsize=14)
ax.set_xlabel('Average Inflation (%)')
ax.set_ylabel('Country')
ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig('../outputs/figures/country_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

print("Figure saved to: outputs/figures/country_comparison.png")

In [ ]:
# Interactive line chart: Price trends by country
fig = px.line(
    df, 
    x='date', 
    y='close', 
    color='country',
    title='Food Price Index by Country Over Time',
    labels={'close': 'Price Index', 'date': 'Date', 'country': 'Country'}
)

fig.update_layout(
    height=600,
    legend=dict(orientation='h', yanchor='bottom', y=-0.3, xanchor='center', x=0.5),
    hovermode='x unified'
)

fig.show()

### 4.4 Correlation Analysis

In [ ]:
# Correlation matrix
correlation_cols = ['open', 'high', 'low', 'close', 'inflation', 'price_range', 'price_change_pct']
correlation_matrix = df[correlation_cols].corr()

# Heatmap
fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(correlation_matrix, dtype=bool))
sns.heatmap(
    correlation_matrix,
    mask=mask,
    annot=True,
    fmt='.3f',
    cmap='RdBu_r',
    center=0,
    square=True,
    linewidths=0.5,
    ax=ax
)
ax.set_title('Correlation Matrix of Price Variables', fontsize=14)

plt.tight_layout()
plt.savefig('../outputs/figures/correlation_matrix.png', dpi=300, bbox_inches='tight')
plt.show()

print("Figure saved to: outputs/figures/correlation_matrix.png")

In [ ]:
# Scatter plot: Price volatility vs Inflation
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Price range vs Inflation
scatter_data = df[df['inflation'].notna()]
axes[0].scatter(scatter_data['price_range'], scatter_data['inflation'], alpha=0.5, s=10, c='steelblue')
axes[0].set_xlabel('Price Range (Volatility)')
axes[0].set_ylabel('Inflation (%)')
axes[0].set_title('Price Volatility vs Inflation')

# Add trend line
z = np.polyfit(scatter_data['price_range'], scatter_data['inflation'], 1)
p = np.poly1d(z)
x_line = np.linspace(scatter_data['price_range'].min(), scatter_data['price_range'].max(), 100)
axes[0].plot(x_line, p(x_line), "r--", alpha=0.8, label=f'Trend line')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Close price vs Inflation
axes[1].scatter(scatter_data['close'], scatter_data['inflation'], alpha=0.5, s=10, c='coral')
axes[1].set_xlabel('Close Price')
axes[1].set_ylabel('Inflation (%)')
axes[1].set_title('Close Price vs Inflation')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../outputs/figures/scatter_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

print("Figure saved to: outputs/figures/scatter_analysis.png")

### 4.5 Seasonal Analysis

In [ ]:
# Average metrics by month
monthly_seasonal = df.groupby('month').agg({
    'close': 'mean',
    'inflation': 'mean',
    'price_range': 'mean'
}).round(3)

# Add month names
month_names = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
monthly_seasonal.index = month_names

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Average close by month
monthly_seasonal['close'].plot(kind='bar', ax=axes[0], color='steelblue', edgecolor='black')
axes[0].set_title('Average Close Price by Month')
axes[0].set_xlabel('Month')
axes[0].set_ylabel('Average Close')
axes[0].tick_params(axis='x', rotation=45)

# Average inflation by month
monthly_seasonal['inflation'].plot(kind='bar', ax=axes[1], color='coral', edgecolor='black')
axes[1].set_title('Average Inflation by Month')
axes[1].set_xlabel('Month')
axes[1].set_ylabel('Average Inflation (%)')
axes[1].tick_params(axis='x', rotation=45)

# Average volatility by month
monthly_seasonal['price_range'].plot(kind='bar', ax=axes[2], color='mediumpurple', edgecolor='black')
axes[2].set_title('Average Price Volatility by Month')
axes[2].set_xlabel('Month')
axes[2].set_ylabel('Average Price Range')
axes[2].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('../outputs/figures/seasonal_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

print("Figure saved to: outputs/figures/seasonal_analysis.png")
print("\nSeasonal Summary:")
monthly_seasonal

---
## 5. Key Insights Summary

### Insights from Exploratory Data Analysis

In [ ]:
# Generate summary insights
print("="*60)
print("KEY INSIGHTS FROM EXPLORATORY DATA ANALYSIS")
print("="*60)

# 1. Overall statistics
print("\n1. OVERALL STATISTICS")
print(f"   - Average close price: {df['close'].mean():.3f}")
print(f"   - Average inflation: {df['inflation'].mean():.2f}%")
print(f"   - Price range: {df['close'].min():.3f} to {df['close'].max():.3f}")
print(f"   - Inflation range: {df['inflation'].min():.2f}% to {df['inflation'].max():.2f}%")

# 2. Top/Bottom countries by inflation
print("\n2. COUNTRIES WITH HIGHEST AVERAGE INFLATION")
top_inflation = df.groupby('country')['inflation'].mean().sort_values(ascending=False).head(5)
for country, inflation in top_inflation.items():
    print(f"   - {country}: {inflation:.2f}%")

print("\n3. COUNTRIES WITH LOWEST AVERAGE INFLATION")
bottom_inflation = df.groupby('country')['inflation'].mean().sort_values().head(5)
for country, inflation in bottom_inflation.items():
    print(f"   - {country}: {inflation:.2f}%")

# 3. Correlation insights
print("\n4. KEY CORRELATIONS")
corr_price_inflation = df['close'].corr(df['inflation'])
corr_volatility_inflation = df['price_range'].corr(df['inflation'])
print(f"   - Price vs Inflation correlation: {corr_price_inflation:.3f}")
print(f"   - Volatility vs Inflation correlation: {corr_volatility_inflation:.3f}")

# 4. Time trends
print("\n5. TRENDS OVER TIME")
first_year_avg = df[df['year'] == df['year'].min()]['close'].mean()
last_year_avg = df[df['year'] == df['year'].max()]['close'].mean()
price_change_total = ((last_year_avg - first_year_avg) / first_year_avg) * 100
print(f"   - Price index change from {df['year'].min()} to {df['year'].max()}: {price_change_total:.1f}%")

# 5. Volatility insights
print("\n6. VOLATILITY ANALYSIS")
most_volatile = df.groupby('country')['price_range'].mean().sort_values(ascending=False).head(3)
print("   Most volatile countries (by avg price range):")
for country, vol in most_volatile.items():
    print(f"   - {country}: {vol:.3f}")

print("\n" + "="*60)

---
## 6. Next Steps

Based on this exploratory analysis, the next steps include:

1. **Hypothesis Testing** (see `Hypothesis_Testing.ipynb`)
   - Test if inflation differs significantly between regions
   - Test correlation significance between price volatility and inflation
   - Test for seasonal patterns in food prices

2. **Predictive Modelling**
   - Time series forecasting for price trends
   - Regression analysis for inflation prediction

3. **Dashboard Development**
   - Create interactive Power BI/Tableau dashboard
   - Enable filtering by country, date range, and metrics

---
## AI Assistance Documentation

This notebook was developed with AI assistance (GitHub Copilot) for:
- Code generation and optimization
- Visualization best practices
- Statistical analysis guidance
- Documentation and comments

All AI-generated code was reviewed, tested, and validated by the analyst.